In [1]:
import joblib
from src import config

model = joblib.load('/Users/ani/Projects/skip_coding_interview/src/artifacts/model.pkl')

# Save Model

In [ ]:
from google.cloud import aiplatform, storage
import joblib
import tempfile
from datetime import datetime

def save_and_register_model(
    model_object,
    model_display_name: str,
    project_id: str,
    location: str,
    gcs_bucket: str,
    alias: str,
    description = None,
    labels = None,
    model_filename: str = "model.pkl",
    is_default_version: bool = False
):
    
    """

    """
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    gcs_model_path = f"models/{model_display_name}/{timestamp}"
    
    storage_client = storage.Client(project=project_id)
    bucket = storage_client.bucket(gcs_bucket)
    
    with tempfile.NamedTemporaryFile(suffix='.pkl') as tmp:
        joblib.dump(model_object, tmp.name)
        blob = bucket.blob(f"{gcs_model_path}/{model_filename}")
        blob.upload_from_filename(tmp.name)
    
    gcs_model_uri = f"gs://{gcs_bucket}/{gcs_model_path}/{model_filename}"
    artifact_uri = f"gs://{gcs_bucket}/{gcs_model_path}"
    
    aiplatform.init(project=project_id, location=location)
    
    existing_models = aiplatform.Model.list(filter=f'display_name="{model_display_name}"')
    
    parent_model = existing_models[0].resource_name if existing_models else None
    vertex_model = aiplatform.Model.upload(
        display_name=model_display_name, 
        artifact_uri=artifact_uri,
        serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest", # plceholder to meet api requirements
        parent_model=parent_model,
        is_default_version=is_default_version,
        version_aliases=[alias],
        version_description=description,
        labels=labels
    )
    
    print(f"Model saved to GCS: {gcs_model_uri}")
    print(f"Model registered: {vertex_model.resource_name}")
    print(f"Alias '{alias}' assigned")
    print(f"Version ID: {vertex_model.version_id}")

In [ ]:
save_and_register_model(
    model_object=model,
    model_display_name='test_model',
    project_id=config.PROJECT_ID,
    location=config.REGION,
    gcs_bucket='jet-bucket-a',
    alias='staging',
    model_filename = "model.pkl"
    )

Creating Model
Create Model backing LRO: projects/266857661098/locations/us-central1/models/1897307369285615616/operations/5045023462402293760
Model created. Resource name: projects/266857661098/locations/us-central1/models/1897307369285615616@8
To use this Model in another session:
model = aiplatform.Model('projects/266857661098/locations/us-central1/models/1897307369285615616@8')
Model saved to GCS: gs://jet-bucket-a/models/test_model/20251228_164106/model.pkl
Model registered: projects/266857661098/locations/us-central1/models/1897307369285615616
Alias 'staging' assigned
Version ID: 8


# Load Model

In [ ]:
from google.cloud import aiplatform, storage
import joblib
import tempfile

def load_model_from_registry(display_name: str, alias: str, project_id: str, region: str):

    aiplatform.init(project=project_id, location=region)
    models = aiplatform.Model.list(filter=f'display_name="{display_name}"')
    
    if not models:
        raise ValueError(f"No model found with display name: {display_name}")
    
    model_id = models[0].name 
    
    vertex_model = aiplatform.Model(model_name=f"{model_id}@{alias}")
    
    print(f"Loaded model: {vertex_model.display_name}")
    print(f"Version ID: {vertex_model.version_id}")
    print(f"Alias: {alias}")
    
    gcs_uri = vertex_model.uri
    storage_client = storage.Client(project=project_id)
    
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as tmp:
        gcs_path = gcs_uri.replace("gs://", "")
        bucket_name = gcs_path.split("/")[0]
        blob_path = "/".join(gcs_path.split("/")[1:])
        
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(f"{blob_path}/model.pkl")
        blob.download_to_filename(tmp.name)
        
        model_object = joblib.load(tmp.name)
    
    return model_object

In [ ]:
encoder = load_model_from_registry("label_encoder", "production", config.PROJECT_ID, config.REGION)
model = load_model_from_registry("trained_model", "production", config.PROJECT_ID, config.REGION)

Loaded model: label_encoder
Version ID: 1
Alias: production
Loaded model: trained_model
Version ID: 1
Alias: production
